# Modellvergleich Ostschweiz (200 Samples)

Vergleich von 4 Modellen auf Ostschweizer Dialektaufnahmen:
- **A**: `openai/whisper-large-v2` → Hochdeutsch (Baseline)
- **B**: `Flurin17/whisper-large-v3-turbo-swiss-german` → Hochdeutsch (fine-tuned)
- **C**: `facebook/wav2vec2-lv-60-espeak-cv-ft` → IPA (sprachunabhängig)
- **D**: `neurlang/ipa-whisper-base` → IPA (Whisper-basiert)

In [ ]:
import os, time, gc, warnings
import pandas as pd
import librosa
import torch
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoModelForSpeechSeq2Seq, AutoProcessor,
    Wav2Vec2ForCTC, Wav2Vec2Processor, pipeline,
)
from IPython.display import display

warnings.filterwarnings("ignore")

CLIPS_DIR = "Data/clips__test"
N_SAMPLES = 200

# Device
if torch.backends.mps.is_available():
    device, torch_dtype = "mps", torch.float32
elif torch.cuda.is_available():
    device, torch_dtype = "cuda:0", torch.float16
else:
    device, torch_dtype = "cpu", torch.float32
print(f"Device: {device}")

## 1. Daten laden – nur Ostschweiz

In [ ]:
df_full = pd.read_csv("Data/test.tsv", sep="\t")
df = df_full[df_full["dialect_region"] == "Ostschweiz"].reset_index(drop=True)

# Auf gewünschte Anzahl begrenzen
if len(df) > N_SAMPLES:
    df = df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)

print(f"Samples: {len(df)} Ostschweiz (von {len(df_full):,} gesamt)")
df.head()

## 2. Hilfsfunktionen

In [ ]:
def free_model():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_audio(path):
    return librosa.load(os.path.join(CLIPS_DIR, path), sr=16000)[0]

## 3. Transkription – sequenziell pro Modell

Jedes Modell wird einzeln geladen, alle Samples transkribiert, dann entladen.

In [ ]:
def transcribe_all(df, transcribe_fn, label):
    """Transkribiert alle Samples mit einer Funktion, zeigt Fortschritt."""
    results = {}
    errors = []
    t0 = time.time()
    total = len(df)
    for i, (_, row) in enumerate(df.iterrows()):
        try:
            y = load_audio(row["path"])
            results[row["path"]] = transcribe_fn(y, row["path"])
        except Exception as e:
            errors.append({"path": row["path"], "error": str(e)})
        if (i + 1) % 50 == 0 or (i + 1) == total:
            elapsed = time.time() - t0
            remain = (elapsed / (i + 1)) * (total - i - 1)
            print(f"  {label}: [{i+1}/{total}] {elapsed/60:.1f}min, ~{remain/60:.1f}min übrig")
    print(f"  → {len(results)} OK, {len(errors)} Fehler\n")
    return results, errors

### 3a. Modell A – Whisper Large V2 (Baseline)

In [ ]:
print("Lade openai/whisper-large-v2 ...")
proc_a = WhisperProcessor.from_pretrained("openai/whisper-large-v2")
model_a = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-large-v2"
).to(device)
model_a.eval()
forced_ids_a = proc_a.get_decoder_prompt_ids(language="german", task="transcribe")

def transcribe_a(y, path):
    feat = proc_a(y, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        ids = model_a.generate(feat, forced_decoder_ids=forced_ids_a)
    return proc_a.batch_decode(ids, skip_special_tokens=True)[0].strip()

results_a, errors_a = transcribe_all(df, transcribe_a, "V2")

del model_a, proc_a
free_model()
print("Modell A entladen.")

### 3b. Modell B – Swiss Whisper (V3 Turbo fine-tuned)

In [ ]:
print("Lade Flurin17/whisper-large-v3-turbo-swiss-german ...")
model_id_b = "Flurin17/whisper-large-v3-turbo-swiss-german"
model_b = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id_b, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
).to(device)
proc_b = AutoProcessor.from_pretrained(model_id_b)
pipe_b = pipeline(
    "automatic-speech-recognition", model=model_b,
    tokenizer=proc_b.tokenizer, feature_extractor=proc_b.feature_extractor,
    torch_dtype=torch_dtype, device=device,
)

def transcribe_b(y, path):
    return pipe_b(os.path.join(CLIPS_DIR, path))["text"].strip()

results_b, errors_b = transcribe_all(df, transcribe_b, "Swiss")

del model_b, proc_b, pipe_b
free_model()
print("Modell B entladen.")

### 3c. Modell C – Wav2Vec2 IPA

In [ ]:
print("Lade facebook/wav2vec2-lv-60-espeak-cv-ft ...")
proc_c = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-lv-60-espeak-cv-ft")
model_c = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-lv-60-espeak-cv-ft").to(device)
model_c.eval()

def transcribe_c(y, path):
    inputs = proc_c(y, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    with torch.no_grad():
        logits = model_c(inputs).logits
    ids = torch.argmax(logits, dim=-1)
    return proc_c.batch_decode(ids)[0].strip()

results_c, errors_c = transcribe_all(df, transcribe_c, "W2V-IPA")

del model_c, proc_c
free_model()
print("Modell C entladen.")

### 3d. Modell D – IPA Whisper Base

In [ ]:
print("Lade neurlang/ipa-whisper-base ...")
proc_d = WhisperProcessor.from_pretrained("neurlang/ipa-whisper-base")
model_d = WhisperForConditionalGeneration.from_pretrained(
    "neurlang/ipa-whisper-base", torch_dtype=torch.float32
).to(device)
model_d.eval()

def transcribe_d(y, path):
    feat = proc_d(y, sampling_rate=16000, return_tensors="pt").input_features.to(device, dtype=model_d.dtype)
    with torch.no_grad():
        ids = model_d.generate(feat)
    return proc_d.batch_decode(ids, skip_special_tokens=True)[0].strip()

results_d, errors_d = transcribe_all(df, transcribe_d, "IPA-Whisper")

del model_d, proc_d
free_model()
print("Modell D entladen.")

## 4. Ergebnisse zusammenführen

In [ ]:
rows = []
for _, row in df.iterrows():
    p = row["path"]
    rows.append({
        "path": p,
        "sentence": row["sentence"],
        "A_v2_hd": results_a.get(p, ""),
        "B_swiss_hd": results_b.get(p, ""),
        "C_w2v_ipa": results_c.get(p, ""),
        "D_whisper_ipa": results_d.get(p, ""),
    })

df_out = pd.DataFrame(rows)
df_out.to_csv("Data/model_comparison_ostschweiz.csv", index=False)
print(f"Gespeichert: {len(df_out)} Zeilen → Data/model_comparison_ostschweiz.csv")
df_out.head(10)

## 5. Vergleich: Hochdeutsch-Pfade (A vs. B)

Wie stark weicht der V2-Output vom Swiss-Whisper-Output ab?

In [ ]:
import Levenshtein

def wer(ref, hyp):
    """Word Error Rate"""
    r = ref.split()
    h = hyp.split()
    if len(r) == 0:
        return float('nan')
    return Levenshtein.distance(ref, hyp) / max(len(ref), 1)

df_out["cer_a_vs_ref"] = df_out.apply(
    lambda r: Levenshtein.distance(r["sentence"], r["A_v2_hd"]) / max(len(r["sentence"]), 1), axis=1)
df_out["cer_b_vs_ref"] = df_out.apply(
    lambda r: Levenshtein.distance(r["sentence"], r["B_swiss_hd"]) / max(len(r["sentence"]), 1), axis=1)

print("CER gegenüber Referenz-Satz (Hochdeutsch):")
print(f"  A (Whisper V2):     Ø {df_out['cer_a_vs_ref'].mean():.3f}")
print(f"  B (Swiss Whisper):  Ø {df_out['cer_b_vs_ref'].mean():.3f}")
print()
better_b = (df_out["cer_b_vs_ref"] < df_out["cer_a_vs_ref"]).sum()
better_a = (df_out["cer_a_vs_ref"] < df_out["cer_b_vs_ref"]).sum()
ties = (df_out["cer_a_vs_ref"] == df_out["cer_b_vs_ref"]).sum()
print(f"  B besser: {better_b}  |  A besser: {better_a}  |  Gleichstand: {ties}")

## 6. Vergleich: IPA-Pfade (C vs. D)

In [ ]:
df_out["cer_c_vs_d"] = df_out.apply(
    lambda r: Levenshtein.distance(r["C_w2v_ipa"], r["D_whisper_ipa"]) / max(len(r["C_w2v_ipa"]), 1)
    if r["C_w2v_ipa"] and r["D_whisper_ipa"] else float('nan'), axis=1)

print("CER zwischen den beiden IPA-Modellen (C vs. D):")
print(f"  Ø CER: {df_out['cer_c_vs_d'].mean():.3f}")
print(f"  Median: {df_out['cer_c_vs_d'].median():.3f}")
print()
print("Beispiel-Vergleich (erste 5):")
for _, r in df_out.head(5).iterrows():
    print(f"  C: {r['C_w2v_ipa'][:60]}")
    print(f"  D: {r['D_whisper_ipa'][:60]}")
    print()

## 7. Visualisierung

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CER-Histogramm A vs B
ax = axes[0]
ax.hist(df_out["cer_a_vs_ref"].dropna(), bins=30, alpha=0.6, label="A (V2)", color="#4C72B0")
ax.hist(df_out["cer_b_vs_ref"].dropna(), bins=30, alpha=0.6, label="B (Swiss)", color="#55A868")
ax.set_xlabel("CER vs. Referenz")
ax.set_ylabel("Anzahl")
ax.set_title("Hochdeutsch-Qualität: V2 vs. Swiss Whisper")
ax.legend()

# Scatter A vs B
ax2 = axes[1]
ax2.scatter(df_out["cer_a_vs_ref"], df_out["cer_b_vs_ref"], alpha=0.4, s=15)
lim = max(df_out["cer_a_vs_ref"].max(), df_out["cer_b_vs_ref"].max()) * 1.05
ax2.plot([0, lim], [0, lim], "k--", lw=1)
ax2.set_xlabel("CER A (V2)")
ax2.set_ylabel("CER B (Swiss)")
ax2.set_title("Head-to-Head (unter Diagonale = B besser)")
ax2.set_aspect("equal")

plt.tight_layout()
plt.show()

## 8. Stichprobe: Alle 4 Outputs nebeneinander

In [ ]:
sample_rows = df_out.sample(n=min(10, len(df_out)), random_state=1)
for _, r in sample_rows.iterrows():
    print(f"📝 Referenz:      {r['sentence']}")
    print(f"🔵 A (V2 HD):     {r['A_v2_hd']}")
    print(f"🟢 B (Swiss HD):  {r['B_swiss_hd']}")
    print(f"🟡 C (W2V IPA):   {r['C_w2v_ipa'][:80]}")
    print(f"🟠 D (Wh. IPA):   {r['D_whisper_ipa'][:80]}")
    print("-" * 80)